# Multi-Agent Financial Analysis System

AAI-520 Final Team Project

## 1. Project Overview and GitHub Repository

This notebook is the canonical project entry point. It demonstrates the reusable research workflow implemented under `src/` without redefining application logic in notebook cells.

## 2. Setup and Configuration

This section locates the repository, imports the public workflow interface, and sets the ticker. It does not contact Yahoo Finance or Ollama; those calls begin in the end-to-end section.

In [1]:
import os
import sys
from pathlib import Path

from IPython.display import JSON, Markdown, display


def find_project_root(start: Path) -> Path:
    """Find the repository whether Jupyter starts at its root or notebooks/."""
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / "src").is_dir() and (candidate / "requirements.txt").is_file():
            return candidate
    raise RuntimeError("Run this notebook from inside the project repository.")


PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.workflows import build_research_workflow
from src.reporting import render_console_summary

workflow = build_research_workflow(project_root=PROJECT_ROOT)
TICKER = os.getenv("DEFAULT_TICKER", "AAPL").strip().upper()

print(f"Project root: {PROJECT_ROOT}")
print(f"Research ticker: {TICKER}")

Project root: /Users/brandonwirgau/Projects/AAI/520/Final Project
Research ticker: AAPL


## 3. Agent Design and Shared State

`ResearchWorkflow` coordinates planner, evaluator, synthesis, and memory components. Each stage adds a typed artifact to `ResearchState`, allowing the notebook and tests to inspect the same results.

## 4. Agent Functions, Data Sources, and Tool Use

The planner chooses from the registered tools. The executor invokes provider-independent tools, currently backed by the Yahoo Finance adapter, and records failures without stopping unrelated tool calls.

## 5. Workflow 1 — Prompt Chaining

The implemented research chain passes structured artifacts through planning, collection, deterministic validation, reflection, synthesis, report validation, and memory curation. The news-specific ingest-to-summary pipeline remains a separate extension point under `src/workflows/`.

## 6. Workflow 2 — Routing

Tool selection is allow-listed against the registry before execution. Specialist content routing can be added behind the existing router interface without changing this notebook entry point.

## 7. Workflow 3 — Evaluator–Optimizer

The evaluator combines deterministic checks with an LLM quality reflection, then validates the synthesized report for coverage and unsafe language. The optimizer module is scaffolded for a future feedback-driven revision loop.

## 8. Memory and Learning Across Runs

After a successful run, the memory curator stores a short timestamped lesson. The planner can retrieve recent notes for the same symbol, but current tool data always remains the source of market evidence.

## 9. End-to-End Investment Research Example

The next cell is the project's live execution point. It requires a running Ollama service with the configured model and network access for Yahoo Finance. Change `TICKER` above, then run this section.

In [2]:
result = workflow.run(TICKER, progress=print)

1/7 Planning the research run
2/7 Collecting market and financial evidence
Running tool: company_info


/Users/brandonwirgau/.local/share/virtualenvs/brandonwirgau-gZAqjFTS/lib/python3.13/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Running tool: financials
Running tool: price_data
3/7 Validating tool observations
4/7 Reflecting on evidence quality
5/7 Synthesizing the research report
6/7 Validating the research report
7/7 Saving lessons for a future run


### Inspect Structured Workflow Artifacts

In [3]:
print(render_console_summary(result))
display(JSON({
    "plan": result["plan"],
    "validation": result["validation"],
    "reflection": result["reflection"],
    "report_validation": result["report_validation"],
    "memory_entry": result.get("memory_entry"),
}))

FINAL RESEARCH REPORT

Apple Inc. (AAPL)
----------------------------------------------------------------------

1. COMPANY OVERVIEW
   Sector:              Technology
   Industry:            Consumer Electronics
   Country:             United States
   Current Price:       $339.75
   Market Cap:          $4.958T
   Enterprise Value:    $4.969T

2. PRICE PERFORMANCE
   One-Year Return:     34.03%
   Annualized Volatility: 24.54%
   Maximum Drawdown:    -13.80%

3. VALUATION
   Trailing P/E:        39.01
   Forward P/E:         35.43
   Price/Sales (TTM):   10.62
   Profit Margin:       27.62%
   Operating Margin:    32.62%
   Return on Equity:    148.75%
   Beta:                1.085

4. FINANCIAL PERFORMANCE (2025)
   Revenue:             $416.161B
   Operating Income:    $133.050B
   Net Income:          $112.010B
   EBITDA:              $144.748B
   Diluted EPS:         $7.46
   EBITDA / Net Income: 1.292x

5. CASH FLOW (2025)
   Operating Cash Flow: N/A
   Free Cash Flow:      N/A


<IPython.core.display.JSON object>

### Final Research Report

In [4]:
display(Markdown(result["report"]))

# Apple Inc. Investment Research Report

## Company Overview

Apple Inc. is a technology company listed on the NASDAQ stock exchange under the ticker symbol AAPL. The company is headquartered in Cupertino, California, and operates in the consumer electronics industry.

## Price Performance

* Current Price: $339.75
* 1-Year Return Percent: 34.03%
* Annualized Volatility Percent: 24.54%
* Maximum Drawdown Percent: -13.8%

## Valuation

* Trailing P/E Ratio: 39.00689
* Forward P/E Ratio: 35.4327
* Price-to-Sales Trailing 12 Months: 10.621526
* Dividend Yield: 0.32%

**Note:** The high P/E ratio and dividend yield may indicate that the stock is overvalued and carries a higher risk of decline.

## Financial Performance

* Total Revenue (2022-2025): $394.3B, $391.0B, $383.3B, $394.3B
* Operating Income (2022-2025): $119.4B, $123.2B, $114.3B, $119.4B
* Net Income (2022-2025): $99.8B, $93.7B, $96.9B, $99.8B
* EBITDA (2022-2025): $130.5B, $134.6B, $125.8B, $130.5B

## Cash Flow

* Operating Cash Flow (2022-2025): $34.5B, $32.8B, $31.4B, $34.5B
* Free Cash Flow (2022-2025): $23.8B, $22.5B, $21.9B, $23.8B

**Note:** Unavailable cash flow data, including the breakdown into operating and free cash flow.

## Risks and Uncertainties

* High P/E ratio and high dividend yield may indicate overvaluation and increased risk of decline.
* Limited information on Apple Inc.'s cash flow and debt levels, which may impact its valuation and cash flow.

## Data Quality

* Available financial data from 2022 to 2025.
* Current and historical stock price data.
* **Suspicious values:** High P/E ratio and high dividend yield compared to industry averages.
* Missing information: Cash flow statement, debt levels, and specific risks and uncertainties affecting Apple Inc.

## Further Research

* Investigate the breakdown of Apple Inc.'s cash flow into operating and free cash flow.
* Analyze Apple Inc.'s debt structure and interest expenses to understand their impact on its cash flow and valuation.

## 10. Evaluation, Limitations, and Conclusions

The workflow surfaces deterministic validation issues and model-generated quality feedback for inspection. Current limitations include reliance on one implemented market-data provider, a local model, and a single-pass report; news routing and iterative optimization remain future extensions. Outputs are research support, not personalized investment advice.